# 02 | Processing a radiometric time series with 3C-O25

This notebook applies the 3C-O25 model to a sequence of above-water radiometric measurements. It uses the repository's maintained time-series runner rather than duplicating the processing code inside the notebook.

By the end, you will have:

- checked the Python environment and repository paths
- loaded and inspected the supplied jetty-format CSV
- verified that every instrument record carries its own wavelength metadata
- reviewed the solar and viewing geometry used by the workflow
- processed the complete time series
- opened the generated NetCDF product
- inspected temporal and spectral behaviour of the retrieved $R_{rs}$ and modeled glint $R_g$
- reviewed practical scientific quality checks and limitations

The central signal decomposition is

$$
\frac{L_t}{E_s} = R_{rs} + R_g,
$$

where $L_t$ is total radiance, $E_s$ is downwelling irradiance, $R_{rs}$ is remote-sensing reflectance, and $R_g$ is the modeled glint contribution.

The fit uses a modeled aquatic term $R_{rs,\mathrm{mod}}$ together with $R_g$ to reproduce measured $L_t/E_s$. The final aquatic spectrum is obtained from the measurement:

$$
R_{rs} = \frac{L_t}{E_s} - R_g.
$$

**Scientific reference:** Pitarch, J. (2026), *A general model for sun and sky glint removal in above-water optical radiometry: mathematical description and Python code*, Earth Science Informatics, 19:78. DOI: 10.1007/s12145-026-02114-w.

## 1. How to run this notebook

This notebook supports both **Visual Studio Code** and **Jupyter Notebook**. Use the repository's `.venv` environment in either application.

### Option A: Visual Studio Code

1. Open the repository folder in Visual Studio Code.
2. Open `notebooks/02_TimeSeries_Processing.ipynb`.
3. Select the kernel in the upper-right corner.
4. Choose `<repository>/.venv/Scripts/python.exe` on Windows, or `<repository>/.venv/bin/python` on Linux and macOS.
5. Select **Restart**, then **Run All**.

Activating `.venv` in a terminal does not automatically select the same interpreter as the notebook kernel. The first code cell prints the actual kernel executable so that it can be verified.

### Option B: Jupyter Notebook

Activate the project environment from the repository root.

On Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
```

On Linux or macOS:

```bash
source .venv/bin/activate
```

Start Jupyter Notebook:

```bash
python -m jupyter notebook
```

Open `notebooks/02_TimeSeries_Processing.ipynb`, select the `.venv` kernel, then choose **Kernel → Restart & Run All**.

### If the environment is not ready

On Windows, run from the repository root:

```powershell
.\tools\setup_project.ps1
```

Otherwise install the project and time-series dependencies with:

```bash
python -m pip install -e ".[timeseries]"
```

> Run cells from top to bottom. Restarting the kernel and running all cells is the best reproducibility check.

## 2. Locate the repository and import the local code

The notebook may start with its working directory set either to the repository root or to `notebooks/`. The next cell searches upward for `pyproject.toml` and `src/rrs3c` before importing the model class.

This ordering is important: repository discovery and path setup must occur **before** `from rrs3c import rrs_model_3C_O25`.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

from rrs3c import rrs_model_3C_O25


def find_repository_root(start: Path) -> Path:
    """Find the nearest parent containing the project source tree."""
    start = start.resolve()

    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (
            folder / "src" / "rrs3c"
        ).is_dir():
            return folder

    raise FileNotFoundError(
        "Could not locate the 3C-Rrs-O25 repository. "
        "Keep this notebook inside the repository and try again."
    )


def load_module_from_file(
    module_name: str,
    module_path: Path,
):
    """Load a Python module from an explicit file path."""
    module_spec = importlib.util.spec_from_file_location(
        module_name,
        module_path,
    )

    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"Could not load the module from: {module_path}")

    module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(module)

    return module


REPO_ROOT = find_repository_root(Path.cwd())

TIMESERIES_DIR = REPO_ROOT / "examples" / "timeseries"
TIMESERIES_DATA_DIR = TIMESERIES_DIR / "data"
TIMESERIES_OUTPUT_DIR = TIMESERIES_DIR / "output"

RUNNER = TIMESERIES_DIR / "src" / "run_timeseries.py"
UTILS_FILE = TIMESERIES_DIR / "src" / "utils.py"

INPUT_FILE = TIMESERIES_DATA_DIR / "example_time_series_data.csv"


if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"The example time-series CSV is missing: {INPUT_FILE}")

if not RUNNER.is_file():
    raise FileNotFoundError(f"The time-series runner is missing: {RUNNER}")

if not UTILS_FILE.is_file():
    raise FileNotFoundError(f"The time-series utility module is missing: {UTILS_FILE}")


timeseries_utils = load_module_from_file(
    module_name="rrs3c_timeseries_utils",
    module_path=UTILS_FILE,
)

angular_difference_deg = timeseries_utils.angular_difference_deg

flags_jetty = timeseries_utils.flags_jetty

interpolate_spectrum = timeseries_utils.interpolate_spectrum

load_jetty_data = timeseries_utils.load_jetty_data

solar_az_el = timeseries_utils.solar_az_el


print(f"Python executable: {sys.executable}")
print(f"Repository root: {REPO_ROOT}")
print(f"Input file: {INPUT_FILE}")
print(f"Runner: {RUNNER}")
print(f"Utility module: {UTILS_FILE}")
print(f"Model class: {rrs_model_3C_O25.__name__}")
print("Notebook setup completed successfully.")

## 3. Import the analysis packages

The processing runner requires NumPy, pandas, Matplotlib, xarray, lmfit, and SciPy. If an import fails, confirm that the notebook kernel points to `.venv` and that the `timeseries` dependency group is installed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("xarray:", xr.__version__)

## 4. Understand the input format

Each row of the jetty-format CSV contains:

1. instrument identifier
2. measurement type
3. timestamp
4. first wavelength $\lambda_1$
5. final wavelength $\lambda_{end}$
6. wavelength increment $\Delta\lambda$
7. spectral values

Relevant measurement types include:

- `ES`: downwelling irradiance $E_s$
- `LSKY-EAST` and `LSKY-WEST`: sky radiance $L_i$
- `LSFC-EAST` and `LSFC-WEST`: total surface-viewing radiance $L_t$

Every row must be interpreted with its **own** wavelength metadata. In the supplied dataset, western surface-radiance records begin at 317 nm, whereas several other streams begin at 320 nm. The maintained utility function handles this difference before interpolation to the common 350–940 nm processing grid.

In [ ]:
(
    instrument,
    variable,
    times,
    wavelength_start,
    wavelength_end,
    wavelength_step,
    spectra,
) = load_jetty_data(INPUT_FILE)

times = pd.DatetimeIndex(pd.to_datetime(times, errors="raise"))

summary = pd.DataFrame(
    {
        "instrument": instrument,
        "variable": variable,
        "time": times,
        "wavelength_start_nm": wavelength_start,
        "wavelength_end_nm": wavelength_end,
        "wavelength_step_nm": wavelength_step,
    }
)

print("Records:", len(summary))
print("Time range:", summary["time"].min(), "to", summary["time"].max())
print("Measurement types:", sorted(summary["variable"].unique()))
display(summary.head(10))

## 5. Verify the per-record spectral ranges

This check makes the wavelength difference visible. Reusing the irradiance wavelength origin for all sensors would shift some spectra and discard valid samples. The corrected workflow instead uses each row's $\lambda_1$, $\lambda_{end}$, and $\Delta\lambda$.

In [ ]:
range_summary = (
    summary.groupby("variable")
    .agg(
        records=("variable", "size"),
        start_min_nm=("wavelength_start_nm", "min"),
        start_max_nm=("wavelength_start_nm", "max"),
        end_min_nm=("wavelength_end_nm", "min"),
        end_max_nm=("wavelength_end_nm", "max"),
        step_min_nm=("wavelength_step_nm", "min"),
        step_max_nm=("wavelength_step_nm", "max"),
    )
    .sort_index()
)
range_summary

## 6. Inspect one acquisition before processing

The common target grid is 350–940 nm at 1 nm spacing. The following cell selects the first timestamp, interpolates the available full-range radiometric streams using their individual metadata, and plots them.

This is an input inspection step, not yet a model fit.

In [ ]:
TARGET_WAVELENGTHS = np.arange(350.0, 941.0, 1.0)
first_time = times.min()
first_mask = np.asarray(times == first_time)


def first_row_for(measurement_type: str) -> int | None:
    rows = np.where((variable == measurement_type) & first_mask)[0]
    return int(rows[0]) if rows.size else None


first_spectra = {}
for measurement_type in [
    "ES",
    "LSKY-EAST",
    "LSFC-EAST",
    "LSKY-WEST",
    "LSFC-WEST",
]:
    row = first_row_for(measurement_type)
    if row is not None:
        first_spectra[measurement_type] = interpolate_spectrum(
            row_index=row,
            l1=wavelength_start,
            l_end=wavelength_end,
            delta_l=wavelength_step,
            spec=spectra,
            target_wavelengths=TARGET_WAVELENGTHS,
        )

print("First acquisition:", first_time)
print("Available streams:", list(first_spectra))

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(TARGET_WAVELENGTHS, first_spectra["ES"], color="tab:orange")
axes[0].set_ylabel("Es (W/m²/nm)")
axes[0].set_title("Interpolated radiometric inputs at the first acquisition")

axes[1].plot(TARGET_WAVELENGTHS, first_spectra["LSKY-EAST"], label="East")
axes[1].plot(TARGET_WAVELENGTHS, first_spectra["LSKY-WEST"], label="West")
axes[1].set_ylabel("Li (W/m²/nm)")
axes[1].legend()

axes[2].plot(TARGET_WAVELENGTHS, first_spectra["LSFC-EAST"], label="East")
axes[2].plot(TARGET_WAVELENGTHS, first_spectra["LSFC-WEST"], label="West")
axes[2].set_ylabel("Lt (W/m²/nm)")
axes[2].set_xlabel("Wavelength (nm)")
axes[2].legend()

for axis in axes:
    axis.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Review quality flags and geometry

The example workflow applies simple irradiance flags and skips observations when:

- $E_s(480) < 20$ (`dark`)
- $E_s(680) > E_s(470)$ (`red`)
- solar elevation is below $5^\circ$

The astronomical helper calculates solar azimuth and elevation for the NIOZ jetty coordinates used by the example. Relative azimuth is the shortest circular separation between the solar and viewing azimuths, constrained to $0^\circ$–$180^\circ$.

The anomaly flag based on $E_s(370)/E_s(940)$ is unavailable on the 350–940 nm processing grid and is therefore reported as `NaN`; it is not used to reject observations in this workflow.

In [ ]:
SITE_LATITUDE_DEG = 53.001788
SITE_LONGITUDE_DEG = 4.789151
EAST_VIEW_AZIMUTH_DEG = 135.0
WEST_VIEW_AZIMUTH_DEG = 225.0

es_first = first_spectra["ES"]
dark, red, anomaly = flags_jetty(TARGET_WAVELENGTHS, es_first)
solar_azimuth_array, solar_elevation_array = solar_az_el(
    first_time.to_pydatetime(),
    lat=SITE_LATITUDE_DEG,
    lon=SITE_LONGITUDE_DEG,
)
solar_azimuth = float(np.asarray(solar_azimuth_array).squeeze())
solar_elevation = float(np.asarray(solar_elevation_array).squeeze())

geometry_summary = pd.DataFrame(
    {
        "value": [
            solar_azimuth,
            solar_elevation,
            angular_difference_deg(solar_azimuth, EAST_VIEW_AZIMUTH_DEG),
            angular_difference_deg(solar_azimuth, WEST_VIEW_AZIMUTH_DEG),
            dark,
            red,
            anomaly,
        ]
    },
    index=[
        "solar azimuth (deg)",
        "solar elevation (deg)",
        "east relative azimuth (deg)",
        "west relative azimuth (deg)",
        "dark flag",
        "red flag",
        "anomaly flag",
    ],
)
geometry_summary

## 8. Run the maintained time-series processor

The notebook calls `examples/timeseries/src/run_timeseries.py` with the **same Python executable as the notebook kernel**. This avoids maintaining a second copy of the scientific processing loop inside the notebook.

The runner:

- loads the CSV
- interpolates each record from its own wavelength metadata
- calculates solar geometry
- fits east and west views independently
- writes selected $R_{rs}$, $R_g$, irradiance, and residual diagnostics to NetCDF

The command below creates `Rrs_timeseries_20200530.nc` under `examples/timeseries/output/`. Existing files with the same name may be replaced.

In [ ]:
OUTPUT_TAG = "20200530"
TIMESERIES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = TIMESERIES_OUTPUT_DIR / f"Rrs_timeseries_{OUTPUT_TAG}.nc"

command = [
    sys.executable,
    str(RUNNER),
    "--input-file",
    INPUT_FILE.name,
    "--input-folder",
    str(TIMESERIES_DATA_DIR),
    "--output-folder",
    str(TIMESERIES_OUTPUT_DIR),
    "--date",
    OUTPUT_TAG,
]

print("Running command:")
print(" ".join(f'"{part}"' if " " in part else part for part in command))

completed = subprocess.run(
    command,
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

assert OUTPUT_FILE.is_file(), f"Expected output was not created: {OUTPUT_FILE}"
print("Created:", OUTPUT_FILE)

## 9. Open and inspect the NetCDF product

xarray presents the dataset as named dimensions, coordinates, variables, and metadata. The principal output fields are:

- `Es`: interpolated downwelling irradiance
- `Rrs_east`, `Rrs_west`: final measurement-derived remote-sensing reflectance
- `Rg_east`, `Rg_west`: modeled glint contribution
- `resid_rmse_east`, `resid_rmse_west`: root-mean-square residual in $L_t/E_s$ space

Missing values indicate times that were not processed for a given view, for example because measurements were incomplete or a quality criterion was not satisfied.

In [ ]:
dataset = xr.open_dataset(OUTPUT_FILE)

display(dataset)


required_dimensions = {
    "time",
    "wavelength",
}

required_coordinates = {
    "time",
    "wavelength",
}

required_variables = {
    "Es",
    "Rrs_east",
    "Rrs_west",
    "Rrs_model_east",
    "Rrs_model_west",
    "Rg_east",
    "Rg_west",
    "fit_success_east",
    "fit_success_west",
}

diagnostic_variables = {
    "rmse_east",
    "rmse_west",
    "residual_550_east",
    "residual_550_west",
}


missing_dimensions = required_dimensions.difference(dataset.sizes)

missing_coordinates = required_coordinates.difference(dataset.coords)

missing_variables = required_variables.difference(dataset.data_vars)

available_diagnostics = diagnostic_variables.intersection(dataset.data_vars)

missing_diagnostics = diagnostic_variables.difference(dataset.data_vars)


if missing_dimensions:
    raise ValueError(
        "The NetCDF product is missing required dimensions: "
        f"{sorted(missing_dimensions)}"
    )

if missing_coordinates:
    raise ValueError(
        "The NetCDF product is missing required coordinates: "
        f"{sorted(missing_coordinates)}"
    )

if missing_variables:
    raise ValueError(
        "The NetCDF product is missing required variables: "
        f"{sorted(missing_variables)}"
    )


output_wavelengths = dataset["wavelength"].to_numpy()


if output_wavelengths.ndim != 1:
    raise ValueError("The wavelength coordinate must be one-dimensional.")

if output_wavelengths.size < 2:
    raise ValueError("The wavelength coordinate must contain at least two values.")

if not np.all(np.isfinite(output_wavelengths)):
    raise ValueError("The wavelength coordinate contains non-finite values.")


wavelength_steps = np.diff(output_wavelengths)


if not np.all(wavelength_steps > 0):
    raise ValueError("The wavelength coordinate must be strictly increasing.")

if not np.allclose(
    wavelength_steps,
    wavelength_steps[0],
    rtol=1e-7,
    atol=1e-10,
):
    raise ValueError("The output wavelength grid is not regularly spaced.")


wavelength_increment = float(wavelength_steps[0])

successful_east_fits = int(dataset["fit_success_east"].sum().item())

successful_west_fits = int(dataset["fit_success_west"].sum().item())


print("NetCDF structure check passed.")

print(
    "Dataset dimensions:",
    {dimension: int(size) for dimension, size in dataset.sizes.items()},
)

print(
    "Wavelength grid: "
    f"{output_wavelengths[0]:.0f} to "
    f"{output_wavelengths[-1]:.0f} nm"
)

print(f"Wavelength increment: " f"{wavelength_increment:g} nm")

print(f"Number of wavelengths: " f"{output_wavelengths.size}")

print(f"Number of acquisition times: " f"{dataset.sizes['time']}")

print(f"Successful EAST fits: " f"{successful_east_fits}")

print(f"Successful WEST fits: " f"{successful_west_fits}")

print(
    "Available diagnostic variables:",
    sorted(available_diagnostics),
)

if missing_diagnostics:
    print(
        "Optional diagnostic variables not present:",
        sorted(missing_diagnostics),
    )

## 10. Plot the time series at selected wavelengths

A single wavelength is useful for viewing temporal structure, although a full scientific assessment must also inspect spectra. The plots below show retrieved $R_{rs}$, modeled $R_g$, and fit RMSE.

A lower RMSE generally indicates tighter agreement between measured and modeled $L_t/E_s$. The paper treats fit quality as a useful practical filter, but not as a quantitative uncertainty estimate or proof of perfect glint separation.

In [ ]:
selected_wavelengths = [443.0, 550.0, 650.0]

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

for wavelength in selected_wavelengths:
    dataset["Rrs_east"].sel(wavelength=wavelength).plot(
        ax=axes[0], marker=".", label=f"East {wavelength:.0f} nm"
    )
    dataset["Rrs_west"].sel(wavelength=wavelength).plot(
        ax=axes[0], marker=".", linestyle="--", label=f"West {wavelength:.0f} nm"
    )
axes[0].set_title("Retrieved remote-sensing reflectance")
axes[0].set_ylabel("Rrs (sr$^{-1}$)")
axes[0].legend(ncol=2)

for wavelength in selected_wavelengths:
    dataset["Rg_east"].sel(wavelength=wavelength).plot(
        ax=axes[1], marker=".", label=f"East {wavelength:.0f} nm"
    )
    dataset["Rg_west"].sel(wavelength=wavelength).plot(
        ax=axes[1], marker=".", linestyle="--", label=f"West {wavelength:.0f} nm"
    )
axes[1].set_title("Modeled glint contribution")
axes[1].set_ylabel("Rg (sr$^{-1}$)")
axes[1].legend(ncol=2)

dataset["rmse_east"].plot(ax=axes[2], marker=".", label="East")
dataset["rmse_west"].plot(ax=axes[2], marker=".", label="West")
axes[2].set_title("Fit residual RMSE")
axes[2].set_ylabel("RMSE (sr$^{-1}$)")
axes[2].set_xlabel("Acquisition time")
axes[2].legend()

for axis in axes:
    axis.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 11. Inspect representative spectra

Time-series plots can hide spectral artefacts. The next cell chooses a time with finite east-view $R_{rs}$ and plots east and west $R_{rs}$ and $R_g$ where available.

Look for:

- discontinuities or isolated spikes
- large negative regions in $R_{rs}$
- residual atmospheric absorption features
- implausible disagreement between east and west views
- glint dominating the total signal in challenging geometry

East and west spectra need not be identical because acquisition geometry, timing, and glint conditions differ.

In [ ]:
valid_east = np.isfinite(dataset["Rrs_east"]).any(dim="wavelength")
valid_times = dataset["time"].where(valid_east, drop=True)

if valid_times.size == 0:
    raise RuntimeError("No valid east-view spectra were found in the output.")

representative_time = valid_times.values[valid_times.size // 2]
print("Representative acquisition:", pd.Timestamp(representative_time))

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for side, linestyle in [("east", "-"), ("west", "--")]:
    rrs = dataset[f"Rrs_{side}"].sel(time=representative_time)
    rg = dataset[f"Rg_{side}"].sel(time=representative_time)
    if np.isfinite(rrs).any():
        axes[0].plot(
            dataset["wavelength"], rrs, linestyle=linestyle, label=side.title()
        )
    if np.isfinite(rg).any():
        axes[1].plot(dataset["wavelength"], rg, linestyle=linestyle, label=side.title())

axes[0].set_title("Retrieved remote-sensing reflectance")
axes[0].set_ylabel("Rrs (sr$^{-1}$)")
axes[0].legend()
axes[1].set_title("Modeled glint contribution")
axes[1].set_ylabel("Rg (sr$^{-1}$)")
axes[1].set_xlabel("Wavelength (nm)")
axes[1].legend()

for axis in axes:
    axis.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 12. Summarize processing coverage

This summary counts timestamps with at least one finite spectral value for each view and reports RMSE statistics. It helps distinguish missing or rejected observations from successful fits.

In [ ]:
coverage_rows = []

for side in ("east", "west"):
    rrs_name = f"Rrs_{side}"
    rmse_name = f"rmse_{side}"
    success_name = f"fit_success_{side}"

    valid_spectra = dataset[rrs_name].notnull().any(dim="wavelength")

    coverage_rows.append(
        {
            "view": side,
            "successful_fits": int(dataset[success_name].sum().item()),
            "valid_spectra": int(valid_spectra.sum().item()),
            "median_rmse": float(dataset[rmse_name].median(skipna=True).item()),
            "maximum_rmse": float(dataset[rmse_name].max(skipna=True).item()),
        }
    )

coverage = pd.DataFrame(coverage_rows).set_index("view")

coverage

## 13. Scientific interpretation and cautions

Use the following hierarchy when assessing a processed time series:

1. **Input completeness:** confirm that $E_s$, $L_i$, and $L_t$ are available for each processed view.
2. **Geometry:** examine solar elevation and relative azimuth. O25 explicitly accounts for relative azimuth, which is important in high-glint, low-azimuth conditions.
3. **Fit quality:** inspect RMSE and residual spectral structure. A close fit is necessary in practice, but is not a complete uncertainty estimate.
4. **Spectral plausibility:** inspect full $R_{rs}$ and $R_g$ spectra, not only selected wavelengths.
5. **Cross-view consistency:** compare east and west results with awareness of their different geometry and timing.
6. **Quality-control provenance:** retain reasons for skipped or failed observations in operational processing.

### Do not over-interpret fitted parameters

The model's aquatic parameters support the construction of a plausible $R_{rs,\mathrm{mod}}$ during fitting. Optical ambiguity means that optimized values such as chlorophyll-like parameter `C` are not automatically validated concentration retrievals. Apply dedicated algorithms to final $R_{rs}$ if geophysical constituent concentrations are required.

### Model scope

The framework explicitly models diffuse and direct glint and can process conditions that simpler low-glint approaches often reject. Low-glint geometry remains preferable for minimizing uncertainty. Whitecaps, foam, bubbles, polarization effects, unusual atmospheres, and high-altitude environments may require additional treatment or adapted assumptions.

## 14. Close the dataset and continue

Close the xarray dataset after inspection so that the NetCDF file is released cleanly.

Next steps:

- use `03_Parameter_Analysis.ipynb` for deeper output diagnostics
- run `examples/timeseries/src/run_timeseries.py --help` for command-line options
- adapt the loader only after documenting a new instrument's row structure, units, timestamps, and wavelength metadata
- preserve the software version, processing configuration, and quality-control decisions with scientific outputs

In [ ]:
dataset.close()
print("Dataset closed.")